In [1]:
# Numerical & data handling
import numpy as np
import pandas as pd
import math

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import LabelEncoder, PowerTransformer, StandardScaler, RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Specialized packages
from ISLP import confusion_table

# Function

In [2]:
def preprocess_split(X, y, train_years=(2014,2022), test_years=(2023,2024)):
    
    # Extract year from index
    years = pd.Series(X.index.astype(str).str[-4:].astype(int), index=X.index)
    train_mask = years.between(*train_years)
    test_mask  = years.between(*test_years)
    
    # Split data
    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]
    
    # Pipeline with median imputation, Yeo-Johnson transform, and scaling
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('yeojohnson', PowerTransformer(method='yeo-johnson')),
        ('standard', StandardScaler())
    ])
    
        # --- Apply imputer separately to raw train/test ---
    imputer = SimpleImputer(strategy='median')
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train),
                               columns=X_train.columns, index=X_train.index)
    X_test_imp = pd.DataFrame(imputer.transform(X_test),
                              columns=X_test.columns, index=X_test.index)
    
    # Fit on train, transform both train and test
    X_tr_sc = pipeline.fit_transform(X_train)
    X_ts_sc = pipeline.transform(X_test)
    
    # Convert back to DataFrame with same columns and index
    X_tr_sc = pd.DataFrame(X_tr_sc, columns=X_train.columns, index=X_train.index)
    X_ts_sc = pd.DataFrame(X_ts_sc, columns=X_test.columns, index=X_test.index)
    
    return X_train_imp, X_test_imp, y_train, y_test, X_tr_sc, X_ts_sc, pipeline


In [3]:
def evaluate_models(models, X_train, y_train, X_test, y_test, model_names=None, label_encoder=None):
    """
    Train and evaluate multiple models, return summary DataFrame of classification metrics.
    
    Parameters:
    - models: list of instantiated sklearn/xgboost classifiers
    - X_train, y_train: training data
    - X_test, y_test: test data
    - model_names: optional list of names for models
    - label_encoder: optional LabelEncoder if needed for XGB
    
    Returns:
    - pd.DataFrame with accuracy, weighted precision, recall, F1
    """
    reports = []
    names = model_names if model_names else [type(m).__name__ for m in models]
    
    for model, name in zip(models, names):
        if isinstance(model, XGBClassifier):
            # Encode labels for XGB
            y_train_enc = label_encoder.fit_transform(y_train)
            y_test_enc = label_encoder.transform(y_test)
            model.fit(X_train, y_train_enc)
            y_pred_enc = model.predict(X_test)
            y_pred = label_encoder.inverse_transform(y_pred_enc)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        reports.append({
            "Model": name,
            "Accuracy": report.get("accuracy", None),
            "Weighted Precision": report["weighted avg"]["precision"],
            "Weighted Recall": report["weighted avg"]["recall"],
            "Weighted F1": report["weighted avg"]["f1-score"]
        })
    
    return pd.DataFrame(reports)


In [4]:

def summarize_reports(report_list, model_names=None):
    """
    Takes a list of classification_report dictionaries and returns a summary DataFrame
    with accuracy, weighted precision, recall, and F1-score.

    Parameters:
    - report_list: list of dicts from classification_report(..., output_dict=True)
    - model_names: optional list of model names (same length as report_list)

    Returns:
    - pd.DataFrame with summary metrics
    """
    summary = []
    for i, report in enumerate(report_list):
        name = model_names[i] if model_names else f"Model_{i+1}"
        summary.append({
            "Model": name,
            "Accuracy": report.get("accuracy", None),
            "Weighted Precision": report["weighted avg"]["precision"],
            "Weighted Recall": report["weighted avg"]["recall"],
            "Weighted F1": report["weighted avg"]["f1-score"]
        })
    df = pd.DataFrame(summary)
    pd.set_option("display.max_columns", None)  # ensures all columns are shown
    pd.set_option("display.width", 1000)        # sets wide enough console width
    return df

In [5]:
def evaluate_kmeans(X_train, k_range=range(2,10)):
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    import matplotlib.pyplot as plt
    
    wcss, sil_scores = [], []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42).fit(X_train)
        wcss.append(kmeans.inertia_)
        sil_scores.append(silhouette_score(X_train, kmeans.labels_))
    
    plt.plot(k_range, wcss, 'bx-'); plt.title("Elbow Method"); plt.show()
    plt.plot(k_range, sil_scores, 'ro-'); plt.title("Silhouette Method"); plt.show()


In [6]:
def classify_by_cluster(X_train, y_train, X_test, y_test, labels_train, labels_test, classifier):
    results = {}
    for cluster_id in sorted(set(labels_train)):
        # Select only rows belonging to this cluster
        mask_train = labels_train == cluster_id
        mask_test = labels_test == cluster_id
        
        # Train classifier on this cluster
        clf = classifier()
        clf.fit(X_train[mask_train], y_train[mask_train])
        
        # Predict within the same cluster
        y_pred = clf.predict(X_test[mask_test])
        
        # Store evaluation metrics
        results[cluster_id] = classification_report(
            y_test[mask_test], y_pred, output_dict=True
        )
    return results

# Data preprocessing and preparation

Here we are trying to tidy up the dataset once again so our models can process them.

In [7]:

# Import dataset
tej = pd.read_csv("tej.csv", index_col = 0)

# Replace '-' with NaN
tej.replace('-', np.nan, inplace=True)

# Drop rows where the target column (last column) == "C"
tej_filtered = tej[tej.iloc[:, -1] != "C"]

# Split into X and y
X = tej_filtered.iloc[:, :-1]
y = tej_filtered.iloc[:, -1]


In [8]:
# Splitting the dataset into training set and testing set.
X_train, X_test, y_train, y_test, X_tr_sc, X_ts_sc, pipeline = preprocess_split(X, y)

# Using the selected predictors from the backward elimination
selected_cols = [
    'total liabilities', 'share capital', 'total capital', 'finance costs',
    'current ratio', 'quick ratio', 'interest expense ratio (B)',
    'total liabilities/total net worth', 'operating profit/paid-in capital ratio',
    'retention ratio'
    ]

In [9]:
## Performing K-means with K = 2.

#  from sklearn.cluster import KMeans

# kmeans = KMeans(n_clusters= 2, random_state= 1)
# kmeans.fit(X_tr_sc)
    
# labels_train = kmeans.labels_
# labels_test = kmeans.predict(X_ts_sc)


In [10]:
# # Making the dataset and export it
# tr_sc_cl = X_tr_sc.copy()
# tr_sc_cl["tcri"] = y_train
# tr_sc_cl["cluster"] = labels_train

# ts_sc_cl = X_ts_sc.copy()
# ts_sc_cl["tcri"] = y_test
# ts_sc_cl["cluster"] = labels_test

In [11]:
# Using the extracted file to be used in clustering.

tr_sc_cl1 = pd.read_csv("k_2_tr_sc_cl1.csv", index_col= 0)
tr_sc_cl2 = pd.read_csv("k_2_tr_sc_cl2.csv", index_col= 0)
ts_sc_cl1 = pd.read_csv("k_2_ts_sc_cl1.csv", index_col= 0)
ts_sc_cl2 = pd.read_csv("k_2_ts_sc_cl2.csv", index_col= 0) # All obs belong to cluster 2 (index 1)

tr_sc_cl1 = tr_sc_cl1.drop("cluster", axis = 1)
tr_sc_cl2 = tr_sc_cl2.drop("cluster", axis = 1)
ts_sc_cl1 = ts_sc_cl1.drop("cluster", axis = 1)
ts_sc_cl2 = ts_sc_cl2.drop("cluster", axis = 1)

X_tr_sc_cl1 = tr_sc_cl1.iloc[:, :-1]
y_tr_cl1 = tr_sc_cl1.iloc[:, -1]
X_tr_sc_cl2 = tr_sc_cl2.iloc[:, :-1]
y_tr_cl2 = tr_sc_cl2.iloc[:, -1]
X_ts_sc_cl1 = ts_sc_cl1.iloc[:, :-1]
y_ts_cl1 = ts_sc_cl1.iloc[:, -1]
X_ts_sc_cl2 = ts_sc_cl2.iloc[:, :-1]
y_ts_cl2 = ts_sc_cl2.iloc[:, -1]

# Proposed model

We are going to propose 1 classification technique, i.e., K-means with 2 clusters, and 3 classification techniques, i.e., Classification and Regression Trees (CART), Random Forest (RF), and XGBoost (Extreme Gradient Boosting). We are going to compare them to our benchmark model, Multinomial Logistic Regression.

In [12]:
from sklearn.tree import DecisionTreeClassifier


models = [
    LogisticRegression(penalty= 'l2', solver= 'saga', max_iter= 10000, random_state= 1),
    DecisionTreeClassifier(random_state=1),
    RandomForestClassifier(n_estimators=500, random_state=1),
    XGBClassifier(n_estimators=500, random_state=1, eval_metric="logloss")
]
model_names = ["Multinomial Logistic Regression", "CART", "RandomForest", "XGBoost"]

In [13]:
# XGBoost needs its categorical response variable to be encoded
le = LabelEncoder()

## Proposed model: No cluster

In [14]:
# Performance metrics computed without cluster-based grouping

summary_no_cluster = evaluate_models(models, 
                                     X_tr_sc[selected_cols], y_train, 
                                     X_ts_sc[selected_cols], y_test, 
                                     model_names=model_names,
                                     label_encoder= le)
print(summary_no_cluster.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.543533            0.537921         0.543533     0.534785
                           CART  0.479290            0.483352         0.479290     0.481030
                   RandomForest  0.618766            0.619810         0.618766     0.618153
                        XGBoost  0.630600            0.633047         0.630600     0.631074


## Proposed model: Cluster 1

In [15]:
# Performance metrics computed for observations in Cluster 1

summary_cl1 = evaluate_models(models, 
                              X_tr_sc_cl1[selected_cols], y_tr_cl1, 
                              X_ts_sc_cl1[selected_cols], y_ts_cl1, 
                              model_names=model_names,
                              label_encoder= le)
print(summary_cl1.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.576087            0.573100         0.576087     0.573274
                           CART  0.489130            0.496807         0.489130     0.491765
                   RandomForest  0.600932            0.601575         0.600932     0.600192
                        XGBoost  0.633540            0.632295         0.633540     0.632224


## Proposed model: Cluster 2

In [ ]:
# # Performance metrics computed for observations in Cluster 2

summary_cl2 = evaluate_models(models, 
                              X_tr_sc_cl2[selected_cols], y_tr_cl2, 
                              X_ts_sc_cl2[selected_cols], y_ts_cl2, 
                              model_names=model_names,
                              label_encoder= le)
print(summary_cl2.to_string(index=False))

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

                          Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
Multinomial Logistic Regression  0.528757            0.523836         0.528757     0.506887
                           CART  0.491651            0.498395         0.491651     0.494425
                   RandomForest  0.614100            0.620334         0.614100     0.612280
                        XGBoost  0.619666            0.618919         0.619666     0.617531


c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

# Summary

In [37]:
# Summary of the results

# Add a column to identify the grouping
summary_no_cluster["Group"] = "No cluster"
summary_cl1["Group"] = "Cluster 1"
summary_cl2["Group"] = "Cluster 2"

# Concatenate into one DataFrame
combined_summary = pd.concat([summary_no_cluster, summary_cl1, summary_cl2],
                             axis=0, ignore_index=True)

# Reorder columns so "Group" comes first
combined_summary = combined_summary[["Group", "Model", "Accuracy", 
                                     "Weighted Precision", "Weighted Recall", "Weighted F1"]].round(2)

print(combined_summary.to_string(index=False))


     Group                           Model  Accuracy  Weighted Precision  Weighted Recall  Weighted F1
No cluster Multinomial Logistic Regression      0.54                0.54             0.54         0.53
No cluster                            CART      0.48                0.48             0.48         0.48
No cluster                    RandomForest      0.62                0.62             0.62         0.62
No cluster                         XGBoost      0.63                0.63             0.63         0.63
 Cluster 1 Multinomial Logistic Regression      0.58                0.57             0.58         0.57
 Cluster 1                            CART      0.49                0.50             0.49         0.49
 Cluster 1                    RandomForest      0.60                0.60             0.60         0.60
 Cluster 1                         XGBoost      0.63                0.63             0.63         0.63
 Cluster 2 Multinomial Logistic Regression      0.53                0.52 

# Previous works

Below are my previous works before I wrap them into functions to make it more presentable. I leave them here for perusal, checking, and transparency. I compared the results of my previous works and with the results after wrapping, to ensure credibility.

## Multinomial Logistic Regression (MLR)

### MLR: No cluster

In [18]:
model_all = LogisticRegression(penalty= 'l2', solver= 'saga', max_iter= 10000, random_state= 1)
results_all = model_all.fit(X_tr_sc,y_train)
y_pred_all = results_all.predict(X_ts_sc)

ct_all = confusion_table(y_pred_all, y_test)
cr_all = classification_report(y_test, y_pred_all)
print(ct_all)
print(cr_all)

Truth        5    6    7   8   9  D
Predicted                          
5          258  130   21   4   1  0
6           76  214   78  14   1  0
7            8   59  113  59  10  0
8            1    5   26  36  11  0
9            2    0    1  19  29  5
D            0    0    0   0   2  0
              precision    recall  f1-score   support

           5       0.62      0.75      0.68       345
           6       0.56      0.52      0.54       408
           7       0.45      0.47      0.46       239
           8       0.46      0.27      0.34       132
           9       0.52      0.54      0.53        54
           D       0.00      0.00      0.00         5

    accuracy                           0.55      1183
   macro avg       0.43      0.43      0.43      1183
weighted avg       0.54      0.55      0.54      1183



### MLR: CLuster 1

In [19]:
model_cl1 = LogisticRegression(penalty= 'l2', solver= 'saga', max_iter= 10000, random_state= 1)
results_cl1 = model_cl1.fit(X_tr_sc_cl1, y_tr_cl1)
y_pred_cl1 = results_cl1.predict(X_ts_sc_cl1)

ct_cl1 = confusion_table(y_pred_cl1, y_ts_cl1)
cr_cl1 = classification_report(y_ts_cl1, y_pred_cl1)
print(ct_cl1)
print(cr_cl1)

Truth        5    6   7   8   9  D
Predicted                         
5          135   59   6   1   1  0
6           27  144  40   7   1  0
7            4   35  67  28   2  0
8            0    3  18  28  11  0
9            0    0   1   7  16  3
D            0    0   0   0   0  0
              precision    recall  f1-score   support

           5       0.67      0.81      0.73       166
           6       0.66      0.60      0.63       241
           7       0.49      0.51      0.50       132
           8       0.47      0.39      0.43        71
           9       0.59      0.52      0.55        31
           D       0.00      0.00      0.00         3

    accuracy                           0.61       644
   macro avg       0.48      0.47      0.47       644
weighted avg       0.60      0.61      0.60       644



c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

### MLR: Cluster 2

In [20]:
model_cl2 = LogisticRegression(penalty= 'l2', solver= 'saga', max_iter= 10000, random_state= 1)
results_cl2 = model_cl2.fit(X_tr_sc_cl2,y_tr_cl2)
y_pred_cl2 = results_cl2.predict(X_ts_sc_cl2)

ct_cl2 = confusion_table(y_pred_cl2, y_ts_cl2)
cr_cl2 = classification_report(y_ts_cl2, y_pred_cl2)
print(ct_cl2)
print(cr_cl2)

Truth        5   6   7   8   9  D
Predicted                        
5          132  60  13   2   0  0
6           41  71  35   6   0  0
7            4  31  47  24   7  0
8            1   5  10  19   3  1
9            1   0   2  10  12  1
D            0   0   0   0   1  0
              precision    recall  f1-score   support

           5       0.64      0.74      0.68       179
           6       0.46      0.43      0.44       167
           7       0.42      0.44      0.43       107
           8       0.49      0.31      0.38        61
           9       0.46      0.52      0.49        23
           D       0.00      0.00      0.00         2

    accuracy                           0.52       539
   macro avg       0.41      0.41      0.40       539
weighted avg       0.51      0.52      0.51       539



## Classification and Regression Trees (CART)

### CART: No cluster

In [21]:
# Initialize CART (Decision Tree)
cart = DecisionTreeClassifier(
    criterion="gini",   # or "entropy"
    max_depth=None,     # you can tune this
    random_state=1
)

# Fit on training data
cart.fit(X_tr_sc[selected_cols], y_train)

# Predict on test data
y_pred_cart = cart.predict(X_ts_sc[selected_cols])

# Feature importance
importances = cart.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_tr_sc[selected_cols].columns,
    'Importance': importances
})

print(importances.sort_values(by="Importance", ascending=False))
print(classification_report(y_test, y_pred_cart))


                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.138723
1                           share capital    0.129425
6              interest expense ratio (B)    0.129314
0                       total liabilities    0.127422
7       total liabilities/total net worth    0.107828
2                           total capital    0.088811
3                           finance costs    0.076462
5                             quick ratio    0.076085
4                           current ratio    0.073298
9                         retention ratio    0.052633
              precision    recall  f1-score   support

           5       0.59      0.56      0.58       345
           6       0.50      0.49      0.49       408
           7       0.39      0.42      0.41       239
           8       0.38      0.41      0.39       132
           9       0.36      0.37      0.36        54
           D       0.20      0.20      0.20         5

    accuracy              

### CART: Cluster 1

In [22]:
# Initialize CART (Decision Tree)
cart = DecisionTreeClassifier(
    criterion="gini",   # or "entropy"
    max_depth=None,     # you can tune this
    random_state=1
)

# Fit on training data
cart.fit(X_tr_sc_cl1[selected_cols], y_tr_cl1)

# Predict on test data
y_pred_cart = cart.predict(X_ts_sc_cl1[selected_cols])

# Feature importance
importances = cart.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_tr_sc_cl1[selected_cols].columns,
    'Importance': importances
})

print(importances.sort_values(by="Importance", ascending=False))
print(classification_report(y_ts_cl1, y_pred_cart))

                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.183899
1                           share capital    0.136590
0                       total liabilities    0.120659
6              interest expense ratio (B)    0.111917
7       total liabilities/total net worth    0.109464
2                           total capital    0.085558
3                           finance costs    0.073637
5                             quick ratio    0.069438
9                         retention ratio    0.055432
4                           current ratio    0.053406
              precision    recall  f1-score   support

           5       0.59      0.54      0.56       166
           6       0.54      0.51      0.52       241
           7       0.38      0.41      0.39       132
           8       0.39      0.49      0.44        71
           9       0.48      0.45      0.47        31
           D       0.00      0.00      0.00         3

    accuracy              

### CART: Cluster 2

In [23]:
# Initialize CART (Decision Tree)
cart = DecisionTreeClassifier(
    criterion="gini",   # or "entropy"
    max_depth=None,     # you can tune this
    random_state=1
)

# Fit on training data
cart.fit(X_tr_sc_cl2[selected_cols], y_tr_cl2)

# Predict on test data
y_pred_cart = cart.predict(X_ts_sc_cl2[selected_cols])

# Feature importance
importances = cart.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_tr_sc_cl2[selected_cols].columns,
    'Importance': importances
})

print(importances.sort_values(by="Importance", ascending=False))
print(classification_report(y_ts_cl2, y_pred_cart))

                               Predictors  Importance
6              interest expense ratio (B)    0.163734
1                           share capital    0.119784
7       total liabilities/total net worth    0.119343
2                           total capital    0.115847
8  operating profit/paid-in capital ratio    0.105247
5                             quick ratio    0.091117
0                       total liabilities    0.084253
3                           finance costs    0.082788
9                         retention ratio    0.061805
4                           current ratio    0.056082
              precision    recall  f1-score   support

           5       0.65      0.62      0.64       179
           6       0.50      0.48      0.49       167
           7       0.36      0.39      0.38       107
           8       0.33      0.33      0.33        61
           9       0.41      0.52      0.46        23
           D       0.00      0.00      0.00         2

    accuracy              

In [24]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import GridSearchCV

# param_grid = {
#     "n_estimators": [100, 300, 500],
#     "max_depth": [None, 10, 20, 30],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4],
#     "max_features": ["sqrt", "log2"]
# }

# rf = RandomForestClassifier(random_state=42)
# grid = GridSearchCV(rf, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
# grid.fit(X_tr_sc, y_train)

# print("Best parameters:", grid.best_params_)
# print("Best CV accuracy:", grid.best_score_)


## Random Forest (RF)

### RF: No cluster

In [25]:
# Using random forest to calculate the importance of each variable toward the response variable
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=350, random_state= 1)
rf.fit(X_tr_sc[selected_cols], y_train)
y_pred_rf = rf.predict(X_ts_sc[selected_cols])
importances = rf.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_tr_sc[selected_cols].columns,
    'Importance': importances
})
print(importances.sort_values(by = 'Importance', ascending= False))
print(classification_report(y_test, y_pred_rf))

                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.136568
6              interest expense ratio (B)    0.116268
0                       total liabilities    0.112916
1                           share capital    0.111798
7       total liabilities/total net worth    0.102119
2                           total capital    0.098426
5                             quick ratio    0.091169
3                           finance costs    0.080591
4                           current ratio    0.078246
9                         retention ratio    0.071900
              precision    recall  f1-score   support

           5       0.71      0.71      0.71       345
           6       0.63      0.64      0.64       408
           7       0.50      0.57      0.53       239
           8       0.63      0.51      0.56       132
           9       0.64      0.56      0.59        54
           D       0.00      0.00      0.00         5

    accuracy              

### RF: Cluster 1

In [26]:
rf = RandomForestClassifier(n_estimators=500, random_state= 1)
rf.fit(X_tr_sc_cl1[selected_cols], y_tr_cl1)
y_pred_rf = rf.predict(X_ts_sc_cl1[selected_cols])
importances = rf.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_ts_sc_cl1[selected_cols].columns,
    'Importance': importances
})

cr_rf_cl1 = ()
print(importances.sort_values(by = 'Importance', ascending= False))
print(classification_report(y_ts_cl1, y_pred_rf))

                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.164345
0                       total liabilities    0.113692
6              interest expense ratio (B)    0.107980
1                           share capital    0.101349
7       total liabilities/total net worth    0.100925
5                             quick ratio    0.089351
2                           total capital    0.089249
4                           current ratio    0.079275
9                         retention ratio    0.078409
3                           finance costs    0.075426
              precision    recall  f1-score   support

           5       0.71      0.67      0.69       166
           6       0.63      0.68      0.65       241
           7       0.46      0.47      0.46       132
           8       0.52      0.49      0.51        71
           9       0.65      0.48      0.56        31
           D       0.00      0.00      0.00         3

    accuracy              

### RF: Cluster 2

In [27]:
rf = RandomForestClassifier(n_estimators=500, random_state= 1)
rf.fit(X_tr_sc_cl2[selected_cols], y_tr_cl2)
y_pred_rf = rf.predict(X_ts_sc_cl2[selected_cols])
importances = rf.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_ts_sc_cl2[selected_cols].columns,
    'Importance': importances
})

cr_rf_cl2 = ()
print(importances.sort_values(by = 'Importance', ascending= False))
print(classification_report(y_ts_cl2, y_pred_rf))

                               Predictors  Importance
6              interest expense ratio (B)    0.139672
7       total liabilities/total net worth    0.115384
8  operating profit/paid-in capital ratio    0.109191
1                           share capital    0.106055
0                       total liabilities    0.102727
5                             quick ratio    0.101992
2                           total capital    0.101641
3                           finance costs    0.084689
4                           current ratio    0.079447
9                         retention ratio    0.059202
              precision    recall  f1-score   support

           5       0.67      0.70      0.68       179
           6       0.57      0.57      0.57       167
           7       0.53      0.64      0.58       107
           8       0.72      0.48      0.57        61
           9       0.75      0.52      0.62        23
           D       0.00      0.00      0.00         2

    accuracy              

c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

## Extreme Gradient Boosting (XGB)

### XGB: No cluster

In [28]:
# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Initialize model (you can tune hyperparameters later)
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"   # avoids warning messages
)

# Fit on training data
xgb.fit(X_tr_sc[selected_cols], y_train_enc)

# Predict on test data
y_pred_xgb = xgb.predict(X_ts_sc[selected_cols])

# Map back after predictions
y_pred_xgb = le.inverse_transform(y_pred_xgb)

# Evaluate
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           5       0.72      0.74      0.73       345
           6       0.65      0.62      0.63       408
           7       0.52      0.61      0.56       239
           8       0.60      0.49      0.54       132
           9       0.63      0.59      0.61        54
           D       0.00      0.00      0.00         5

    accuracy                           0.63      1183
   macro avg       0.52      0.51      0.51      1183
weighted avg       0.64      0.63      0.63      1183



### XGB: Cluster 1

In [29]:
# Encode labels
le = LabelEncoder()
y_train_enc1 = le.fit_transform(y_tr_cl1)
y_test_enc1 = le.transform(y_ts_cl1)

# Initialize model (you can tune hyperparameters later)
xgb = XGBClassifier(
    n_estimators=500,
    random_state=1,
    eval_metric="logloss"   # avoids warning messages
)

# Fit on training data
xgb.fit(X_tr_sc_cl1[selected_cols], y_train_enc1)

# Predict on test data
y_pred_xgb = xgb.predict(X_ts_sc_cl1[selected_cols])

# Map back after predictions
y_pred_xgb = le.inverse_transform(y_pred_xgb)

# Evaluate
print(classification_report(y_ts_cl1, y_pred_xgb))

              precision    recall  f1-score   support

           5       0.70      0.77      0.73       166
           6       0.66      0.64      0.65       241
           7       0.52      0.51      0.51       132
           8       0.62      0.58      0.60        71
           9       0.63      0.55      0.59        31
           D       0.25      0.33      0.29         3

    accuracy                           0.63       644
   macro avg       0.56      0.56      0.56       644
weighted avg       0.63      0.63      0.63       644



### XGB: Cluster 2

In [30]:
# Encode labels
le = LabelEncoder()
y_train_enc2 = le.fit_transform(y_tr_cl2)
y_test_enc2 = le.transform(y_ts_cl2)

# Initialize model (you can tune hyperparameters later)
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"   # avoids warning messages
)

# Fit on training data
xgb.fit(X_tr_sc_cl2[selected_cols], y_train_enc2)

# Predict on test data
y_pred_xgb = xgb.predict(X_ts_sc_cl2[selected_cols])

# Map back after predictions
y_pred_xgb = le.inverse_transform(y_pred_xgb)

# Evaluate
print(classification_report(y_ts_cl2, y_pred_xgb))

              precision    recall  f1-score   support

           5       0.71      0.70      0.71       179
           6       0.57      0.60      0.59       167
           7       0.58      0.62      0.60       107
           8       0.60      0.46      0.52        61
           9       0.61      0.61      0.61        23
           D       0.00      0.00      0.00         2

    accuracy                           0.62       539
   macro avg       0.51      0.50      0.50       539
weighted avg       0.62      0.62      0.62       539



c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\NITRO 5\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

# Proposed model: Cluster-then-classify with K = 2

In [31]:
# To maintain the reproducibility, we extract the clustering into csv and reuse it.
# tr_sc_cl

In [32]:
# To maintain the reproducibility, we extract the clustering into csv and reuse it.
# ts_sc_cl

In [33]:
# Initialize CART (Decision Tree)
cart = DecisionTreeClassifier(
    criterion="gini",   # or "entropy"
    max_depth=None,     # you can tune this
    random_state=1
)

# Fit on training data
cart.fit(X_tr_sc[selected_cols], y_train)

# Predict on test data
y_pred_cart = cart.predict(X_ts_sc[selected_cols])

# Feature importance
importances = cart.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_tr_sc[selected_cols].columns,
    'Importance': importances
})

print(importances.sort_values(by="Importance", ascending=False))
print(classification_report(y_test, y_pred_cart))


                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.138723
1                           share capital    0.129425
6              interest expense ratio (B)    0.129314
0                       total liabilities    0.127422
7       total liabilities/total net worth    0.107828
2                           total capital    0.088811
3                           finance costs    0.076462
5                             quick ratio    0.076085
4                           current ratio    0.073298
9                         retention ratio    0.052633
              precision    recall  f1-score   support

           5       0.59      0.56      0.58       345
           6       0.50      0.49      0.49       408
           7       0.39      0.42      0.41       239
           8       0.38      0.41      0.39       132
           9       0.36      0.37      0.36        54
           D       0.20      0.20      0.20         5

    accuracy              

In [34]:
rf = RandomForestClassifier(n_estimators=500, random_state= 1)
rf.fit(X_tr_sc[selected_cols], y_train)
y_pred_rf = rf.predict(X_ts_sc[selected_cols])
importances = rf.feature_importances_

importances = pd.DataFrame({
    'Predictors': X_ts_sc[selected_cols].columns,
    'Importance': importances
})

cr_rf_cl1 = ()
print(importances.sort_values(by = 'Importance', ascending= False))
print(classification_report(y_test, y_pred_rf))

                               Predictors  Importance
8  operating profit/paid-in capital ratio    0.136541
6              interest expense ratio (B)    0.116645
0                       total liabilities    0.113759
1                           share capital    0.111605
7       total liabilities/total net worth    0.101979
2                           total capital    0.098002
5                             quick ratio    0.091235
3                           finance costs    0.080674
4                           current ratio    0.078290
9                         retention ratio    0.071270
              precision    recall  f1-score   support

           5       0.72      0.70      0.71       345
           6       0.61      0.64      0.63       408
           7       0.50      0.55      0.52       239
           8       0.61      0.50      0.55       132
           9       0.63      0.54      0.58        54
           D       0.00      0.00      0.00         5

    accuracy              

In [35]:
# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Initialize model (you can tune hyperparameters later)
xgb = XGBClassifier(
    n_estimators=500,
    random_state=1,
    eval_metric="logloss"   # avoids warning messages
)

# Fit on training data
xgb.fit(X_tr_sc[selected_cols], y_train_enc)

# Predict on test data
y_pred_xgb = xgb.predict(X_ts_sc[selected_cols])

# Map back after predictions
y_pred_xgb = le.inverse_transform(y_pred_xgb)

# Evaluate
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           5       0.71      0.72      0.72       345
           6       0.66      0.63      0.64       408
           7       0.51      0.58      0.54       239
           8       0.57      0.53      0.55       132
           9       0.66      0.57      0.61        54
           D       0.25      0.20      0.22         5

    accuracy                           0.63      1183
   macro avg       0.56      0.54      0.55      1183
weighted avg       0.63      0.63      0.63      1183



In [36]:
import numpy as np
print("Train cluster counts:", np.bincount(labels_train))
print("Test cluster counts:", np.bincount(labels_test))


NameError: name 'labels_train' is not defined

# This is me trying something else

In [ ]:
from sklearn.cluster import KMeans

def cluster_data(X_train, X_test, n_clusters=5, random_state=42):
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_train)
    
    labels_train = kmeans.labels_
    labels_test = kmeans.predict(X_test)
    
    return labels_train, labels_test, kmeans

In [ ]:
from sklearn.metrics import classification_report

def classify_by_cluster(X_train, y_train, X_test, y_test, labels_train, labels_test, classifier):
    rows = []
    results = {}
    for cluster_id in sorted(set(labels_test)):
        # Select only rows belonging to this cluster
        mask_train = labels_train == cluster_id
        mask_test = labels_test == cluster_id
        
        # Train classifier on this cluster
        clf = classifier()
        clf.fit(X_train[mask_train], y_train[mask_train])
        
        # Predict within the same cluster
        y_pred = clf.predict(X_test[mask_test])
        
        # Get classification report
        report = classification_report(y_test[mask_test], y_pred, output_dict=True)

        # Store evaluation metrics in dict
        results[cluster_id] = report
    # Extract key metrics (macro avg or weighted avg)
        rows.append({
            "Cluster": cluster_id,
            "Test Size": mask_test.sum(),
            "Accuracy": report["accuracy"],
            "Precision (macro)": report["macro avg"]["precision"],
            "Recall (macro)": report["macro avg"]["recall"],
            "F1 (macro)": report["macro avg"]["f1-score"]
        })
    
    # Convert to DataFrame for readability
    results_df = pd.DataFrame(rows)
    return results_df

In [ ]:
labels_train, labels_test, kmeans = cluster_data(X_tr_sc, X_ts_sc, n_clusters= 2)
labels_test

In [ ]:
def classify_by_cluster_table(X_train, y_train, X_test, y_test, labels_train, labels_test, classifier):
    """
    Train and evaluate classifiers within each cluster, return results as a clean DataFrame.
    """
    rows = []
    for cluster_id in sorted(set(labels_test)):
        mask_train = labels_train == cluster_id
        mask_test = labels_test == cluster_id
        
        if mask_test.sum() == 0:
            continue  # Skip clusters with no test samples

        clf = classifier()
        clf.fit(X_train[mask_train], y_train[mask_train])
        y_pred = clf.predict(X_test[mask_test])
        
        # Get classification report as dict
        report = classification_report(y_test[mask_test], y_pred, output_dict=True)
        
        # Extract key metrics (macro avg or weighted avg)
        rows.append({
            "Cluster": cluster_id,
            "Test Size": mask_test.sum(),
            "Accuracy": report["accuracy"],
            "Precision (macro)": report["macro avg"]["precision"],
            "Recall (macro)": report["macro avg"]["recall"],
            "F1 (macro)": report["macro avg"]["f1-score"]
        })
    
    # Convert to DataFrame for readability
    results_df = pd.DataFrame(rows)
    return results_df


In [ ]:
results_df = classify_by_cluster_table(X_tr_sc, y_train, X_ts_sc, y_test, labels_train= labels_train, labels_test= labels_test, classifier= RandomForestClassifier)

In [ ]:
results = classify_by_cluster(X_tr_sc, y_train, X_ts_sc, y_test, labels_train= labels_train, labels_test= labels_test, classifier= RandomForestClassifier)

In [ ]:
print(results)

In [ ]:
print(results_df)

In [ ]:
import numpy as np
print("Train cluster counts:", np.bincount(labels_train))
print("Test cluster counts:", np.bincount(labels_test))


In [ ]:
labels_test